# 📊 Azure SQL Advisor — Data Collector

**Notebook 1 of 2** — Incremental telemetry extraction from Azure SQL Database to Azure Storage Account (ADLS Gen2).

This notebook:
1. Connects to Azure SQL Database via JDBC.
2. Reads the last watermark from `_metadata/watermarks.json` in ADLS Gen2.
3. Extracts **incremental** time-series DMVs (`sys.dm_db_resource_stats`, Query Store) and **snapshot** catalog metrics.
4. Writes partitioned Parquet files to `raw/{server}/{database}/{metric}/year=YYYY/month=MM/day=DD/`.
5. Updates watermarks atomically on completion.


In [ ]:
# Databricks notebook source
# MAGIC %md ### ⚙️ Configuration Widgets


In [ ]:
# Create interactive widgets
dbutils.widgets.text("server_name", "", "Azure SQL Server FQDN")
dbutils.widgets.text("database_name", "", "Database Name")
dbutils.widgets.text("storage_account", "", "Storage Account Name")
dbutils.widgets.text("storage_container", "azure-sql-telemetry", "Container Name")
dbutils.widgets.dropdown("auth_method", "managed_identity", ["managed_identity", "sql", "service_principal"], "Authentication Method")
dbutils.widgets.text("sql_username", "", "SQL Username (if SQL auth)")
dbutils.widgets.text("sql_password", "", "SQL Password (if SQL auth)")

server_name = dbutils.widgets.get("server_name")
database_name = dbutils.widgets.get("database_name")
storage_account = dbutils.widgets.get("storage_account")
storage_container = dbutils.widgets.get("storage_container")
auth_method = dbutils.widgets.get("auth_method")
sql_username = dbutils.widgets.get("sql_username")
sql_password = dbutils.widgets.get("sql_password")

print(f"Server: {server_name}")
print(f"Database: {database_name}")
print(f"Storage: {storage_account}/{storage_container}")
print(f"Auth: {auth_method}")


### 📦 Import Configuration


In [ ]:
import sys, os, json
from datetime import datetime, timezone, timedelta

# Add the repo root to PYTHONPATH so config.py can be imported
repo_path = os.path.dirname(os.path.abspath(globals().get('__file__', '/Workspace/Repos/azure_sql_advisor')))
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

from config import (
    AdvisorConfig, INCREMENTAL_QUERIES, SNAPSHOT_QUERIES, ALL_METRICS
)

# Build config from widget values
cfg = AdvisorConfig(
    server=server_name,
    database=database_name,
    storage_account_name=storage_account,
    storage_container=storage_container,
    auth_method=auth_method,
    username=sql_username,
    password=sql_password,
)

# ABFSS base path
ABFSS_BASE = f"abfss://{storage_container}@{storage_account}.dfs.core.windows.net"
RAW_BASE = f"{ABFSS_BASE}/{cfg.storage_base_path}/{server_name}/{database_name}"
WATERMARK_PATH = f"{ABFSS_BASE}/{cfg.watermark_blob_name}"

print(f"ABFSS Base: {ABFSS_BASE}")
print(f"Raw Base: {RAW_BASE}")
print(f"Watermark Path: {WATERMARK_PATH}")


### 🔗 JDBC Connection Setup


In [ ]:
# Build JDBC URL for Azure SQL Database
jdbc_url = f"jdbc:sqlserver://{server_name}:1433;database={database_name};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30"

if auth_method == "managed_identity":
    # Use Azure AD Managed Identity (Databricks-managed)
    jdbc_properties = {
        "authentication": "ActiveDirectoryMSI",
        "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    }
elif auth_method == "service_principal":
    # Service Principal authentication via AAD token
    jdbc_properties = {
        "authentication": "ActiveDirectoryServicePrincipal",
        "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    }
else:
    # SQL Authentication
    jdbc_properties = {
        "user": sql_username,
        "password": sql_password,
        "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    }

print(f"JDBC URL: {jdbc_url[:80]}...")
print(f"Auth mode: {auth_method}")


### 🔖 Watermark Management

Watermarks track the last-extracted timestamp per incremental metric, enabling safe restarts and deduplication.


In [ ]:
def read_watermarks():
    """Read watermarks from ADLS Gen2 JSON file."""
    try:
        content = dbutils.fs.head(WATERMARK_PATH, 65536)
        return json.loads(content)
    except Exception as e:
        print(f"No existing watermarks found ({e}). Starting fresh.")
        return {}


def save_watermarks(watermarks: dict):
    """Write watermarks atomically to ADLS Gen2."""
    import tempfile
    local_tmp = tempfile.mktemp(suffix=".json")
    with open(local_tmp, "w") as f:
        json.dump(watermarks, f, indent=2, default=str)
    dbutils.fs.cp(f"file:{local_tmp}", WATERMARK_PATH)
    os.remove(local_tmp)
    print(f"Watermarks saved: {json.dumps(watermarks, indent=2, default=str)}")


# Load current watermarks
watermarks = read_watermarks()
print(f"Current watermarks: {json.dumps(watermarks, indent=2, default=str)}")


### ⏱️ Incremental Time-Series Extraction

Time-series DMVs (`sys.dm_db_resource_stats`, Query Store) are extracted using watermark-based `WHERE` clauses.


In [ ]:
now = datetime.now(timezone.utc)
new_watermarks = dict(watermarks)
extraction_summary = {}

for metric_name, query_def in INCREMENTAL_QUERIES.items():
    print(f"\n{'='*60}")
    print(f"Extracting incremental metric: {metric_name}")
    print(f"{'='*60}")
    
    try:
        wm_col = query_def['watermark_column']
        current_wm = watermarks.get(metric_name)
        
        # Build WHERE clause based on watermark
        if current_wm:
            if 'query_store' in metric_name:
                where_clause = f"WHERE rs.{wm_col} > '{current_wm}'"
            else:
                where_clause = f"WHERE {wm_col} > '{current_wm}'"
            print(f"  Watermark: {wm_col} > {current_wm}")
        else:
            where_clause = ""
            print(f"  No watermark — full extraction.")
        
        sql = query_def['sql'].format(where_clause=where_clause)
        
        # Execute via PySpark JDBC
        df = spark.read.jdbc(
            url=jdbc_url,
            table=f"({sql}) AS subquery",
            properties=jdbc_properties
        )
        
        record_count = df.count()
        print(f"  Records extracted: {record_count}")
        
        if record_count > 0:
            # Add metadata columns
            from pyspark.sql.functions import lit, current_timestamp
            df = (df
                  .withColumn("server_name", lit(server_name))
                  .withColumn("database_name", lit(database_name))
                  .withColumn("ingestion_time", current_timestamp())
            )
            
            # Write partitioned Parquet
            year = now.strftime("%Y")
            month = now.strftime("%m")
            day = now.strftime("%d")
            output_path = f"{RAW_BASE}/{metric_name}/year={year}/month={month}/day={day}"
            
            df.write.mode("append").parquet(output_path)
            print(f"  Written to: {output_path}")
            
            # Update watermark to the max value of the watermark column
            max_wm = df.agg({wm_col: "max"}).collect()[0][0]
            if max_wm:
                new_watermarks[metric_name] = str(max_wm)
                print(f"  New watermark: {max_wm}")
        
        extraction_summary[metric_name] = {"status": "success", "records": record_count}
        
    except Exception as e:
        print(f"  ERROR: {e}")
        extraction_summary[metric_name] = {"status": "error", "error": str(e)}


### 📸 Snapshot Metric Extraction

Catalog views, wait stats, index stats, and structural metadata are captured as full snapshots each run.


In [ ]:
for metric_name, sql_template in SNAPSHOT_QUERIES.items():
    print(f"\n{'='*60}")
    print(f"Extracting snapshot metric: {metric_name}")
    print(f"{'='*60}")
    
    try:
        # Format SQL template with config values
        sql = sql_template.format(
            top_queries_count=cfg.top_queries_count,
            min_execution_count=cfg.min_execution_count,
            missing_index_impact_threshold=cfg.missing_index_impact_threshold,
            min_index_pages=cfg.min_index_pages,
            fragmentation_reorg_pct=cfg.fragmentation_reorg_pct,
        )
        
        # Execute via PySpark JDBC
        df = spark.read.jdbc(
            url=jdbc_url,
            table=f"({sql}) AS subquery",
            properties=jdbc_properties
        )
        
        record_count = df.count()
        print(f"  Records extracted: {record_count}")
        
        if record_count > 0:
            from pyspark.sql.functions import lit, current_timestamp
            df = (df
                  .withColumn("server_name", lit(server_name))
                  .withColumn("database_name", lit(database_name))
                  .withColumn("ingestion_time", current_timestamp())
            )
            
            # Write partitioned Parquet to ADLS Gen2
            year = now.strftime("%Y")
            month = now.strftime("%m")
            day = now.strftime("%d")
            output_path = f"{RAW_BASE}/{metric_name}/year={year}/month={month}/day={day}"
            
            df.write.mode("overwrite").parquet(output_path)
            print(f"  Written to: {output_path}")
        
        extraction_summary[metric_name] = {"status": "success", "records": record_count}
        
    except Exception as e:
        print(f"  ERROR: {e}")
        extraction_summary[metric_name] = {"status": "error", "error": str(e)}


### 💾 Save Watermarks & Summary


In [ ]:
# Save updated watermarks
new_watermarks["last_successful_run"] = now.isoformat()
save_watermarks(new_watermarks)

# Print extraction summary
print("\n" + "=" * 60)
print("EXTRACTION SUMMARY")
print("=" * 60)

total_records = 0
for metric, info in extraction_summary.items():
    status_icon = "✅" if info["status"] == "success" else "❌"
    records = info.get("records", 0)
    total_records += records
    error_msg = f" — {info.get('error', '')}" if info["status"] == "error" else ""
    print(f"  {status_icon} {metric}: {records} records{error_msg}")

print(f"\nTotal records extracted: {total_records}")
print(f"Watermarks updated: {json.dumps(new_watermarks, indent=2, default=str)}")

# Set notebook exit value for orchestration
dbutils.notebook.exit(json.dumps({
    "status": "success",
    "total_records": total_records,
    "metrics_collected": len(extraction_summary),
    "errors": [m for m, i in extraction_summary.items() if i["status"] == "error"],
    "run_timestamp": now.isoformat(),
}))
